# Love Landscape — Research Analysis

Analysis of anonymous landscape submissions. Connects to Supabase using the **service role key** (never share this key).

## Setup
```bash
pip install -r requirements.txt
cp .env.example .env  # then fill in your keys
```

In [ ]:
import os
from dotenv import load_dotenv
from supabase import create_client
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import numpy as np

load_dotenv()

url = os.environ['SUPABASE_URL']
key = os.environ['SUPABASE_SERVICE_ROLE_KEY']
client = create_client(url, key)

print('Connected to Supabase')

## 1. Load Data

In [ ]:
response = client.table('submissions').select('*').execute()
df = pd.DataFrame(response.data)
print(f'{len(df)} submissions loaded')
df.head()

In [ ]:
# Parameter columns
param_cols = [
    'p_deep_friendships', 'p_romantic_love', 'p_tender_middle',
    'p_casual_touch', 'p_empty_phys_barrier', 'p_ungrounded_barrier',
    'p_uncertainty_tolerance', 'p_openness', 'p_mapped'
]

param_labels = [
    'Friendship depth', 'Romantic bond', 'Tender middle',
    'Physical comfort', 'Grounding need', 'Structure need',
    'Ambiguity comfort', 'Openness', 'Self-knowledge'
]

df[param_cols].describe()

## 2. Parameter Distributions

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
fig.suptitle('Parameter Distributions', fontsize=16, y=1.02)

for i, (col, label) in enumerate(zip(param_cols, param_labels)):
    ax = axes[i // 3, i % 3]
    ax.hist(df[col], bins=20, alpha=0.7, color='#7F77DD', edgecolor='white')
    ax.set_title(label, fontsize=11)
    ax.set_xlim(0, 1)
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

## 3. Correlation Matrix

In [ ]:
corr = df[param_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr,
    xticklabels=param_labels,
    yticklabels=param_labels,
    annot=True, fmt='.2f',
    cmap='RdBu_r', center=0,
    ax=ax
)
ax.set_title('Parameter Correlations')
plt.tight_layout()
plt.show()

## 4. PCA — Dimensionality Reduction

In [ ]:
X = df[param_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f'Explained variance: PC1={pca.explained_variance_ratio_[0]:.2%}, PC2={pca.explained_variance_ratio_[1]:.2%}')
print(f'Total: {sum(pca.explained_variance_ratio_):.2%}')

# Loadings
loadings = pd.DataFrame(
    pca.components_.T,
    index=param_labels,
    columns=['PC1', 'PC2']
)
print('\nPCA Loadings:')
loadings

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.4, s=20, color='#7F77DD')

# Loading vectors
scale = 3
for i, label in enumerate(param_labels):
    ax.annotate(
        label,
        xy=(pca.components_[0, i] * scale, pca.components_[1, i] * scale),
        fontsize=9, ha='center', color='#f97066',
        arrowprops=dict(arrowstyle='->', color='#f97066', lw=0.8),
        xytext=(pca.components_[0, i] * scale * 1.3, pca.components_[1, i] * scale * 1.3)
    )

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title('Landscape Population — PCA')
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
plt.tight_layout()
plt.show()

## 5. Clustering — Landscape Archetypes

In [ ]:
# Elbow method
inertias = []
K_range = range(2, 8)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertias, 'o-', color='#7F77DD')
plt.xlabel('k')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.show()

In [ ]:
# Fit with chosen k (adjust based on elbow)
k = 4
km = KMeans(n_clusters=k, random_state=42, n_init=10)
df['cluster'] = km.fit_predict(X_scaled)

# Plot in PCA space
colors = ['#2dd4a8', '#7F77DD', '#f97066', '#f472b6', '#a78bfa']
fig, ax = plt.subplots(figsize=(10, 8))
for c in range(k):
    mask = df['cluster'] == c
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], alpha=0.5, s=25, color=colors[c], label=f'Cluster {c}')

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title(f'Landscape Archetypes (k={k})')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cluster centroids (original scale)
centroids_scaled = km.cluster_centers_
centroids = scaler.inverse_transform(centroids_scaled)

centroid_df = pd.DataFrame(centroids, columns=param_labels)
centroid_df.index.name = 'Cluster'
centroid_df.round(2)

## 6. Demographic Cross-tabs

In [ ]:
# Mean params by relationship structure
if 'relationship_structure' in df.columns:
    rel_means = df.groupby('relationship_structure')[param_cols].mean()
    rel_means.columns = param_labels
    print('Mean parameters by relationship structure:')
    display(rel_means.round(2))

# Mean params by age range
if 'age_range' in df.columns:
    age_means = df.groupby('age_range')[param_cols].mean()
    age_means.columns = param_labels
    print('\nMean parameters by age range:')
    display(age_means.round(2))

## 7. Export

In [ ]:
df.to_csv('submissions_export.csv', index=False)
print(f'Exported {len(df)} rows to submissions_export.csv')